In diesem Jupyter Notebook wird die Häufigkeit eines spezifischen Wortes in Bundestagsreden über verschiedene Jahre hinweg analysiert und in einem Graph visualisiert. Ein interaktives Widget ermöglicht es dem User ein Token einzugeben und sich die Anzahl der Token pro Jahr anzeigen zu lassen. Der Graph passt sich je nach Input an.

Dieses Notebook wurde von folgendem Artikel der Zeit inspiriert: https://www.zeit.de/kultur/2019-09/beleidigungen-bundestagsreden-arsch-arschloch-bundeswoerter

Hervorzuheben ist die Library Seaborn, ohne die ich den Graph nicht so designen hätte könne, wie ich es mir gewünscht habe. Die Library wurde zur Verbesserung des Stils und des Aussehens des Plots verwendet. Sie stellt ansprechende Standartstile, Farbpaletten und Schriftgrößen bereit und erleichtert die Erstellung und Anpassung des Graphen.

In [1]:
import re
import os
import json
import glob
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ipywidgets import interact, widgets, fixed
from tqdm import tqdm
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from datetime import datetime
from scipy.sparse import csr_matrix, vstack
from tqdm import tqdm
from collections import defaultdict

Hier wird die Funktion zum Laden der Daten definiert und anschließend alle Daten der geparsten Protokolle geladen. Diese wollen wir später analysieren.

In [19]:
def load_text_from_file(filepath):
    with open(filepath, 'r', encoding='utf-8') as f:
        data = json.load(f)
    out_text = ''
    for speech in data:
        out_text+= speech['text']
    return out_text

all_files = glob.glob(os.path.join('/var/lib/private/simonjaned87761/team-04_deprecated/data/opendata_api/parsed_protocols', "*.json"))
text_list = []
for i, file in enumerate(tqdm(all_files)):
    text_list.append((file[-15:-5], load_text_from_file(file)))
    

100%|██████████| 2075/2075 [00:18<00:00, 113.44it/s]


Hier betrachten wir nur die Protokolle von 1991 bis 2024 (exklusive 2025). Ich iteriere einmal über alle Protokolle, um das Jahr zu extrahieren. Da sie das Datum im Namen haben, kann man das Jahr ganz einfach extrahieren. Anschließend werden die Protokolle nach Jahren gruppiert.

In [21]:
years = range(1991,2025)
grouped_years = {str(year):[] for year in years}
for date_str, text in text_list:
    year = str(datetime.strptime(date_str,'%Y-%m-%d' ).year)
    grouped_years[year].append((date_str, text))
    print(f"appended ({date_str}, {text[:10]}) to grouped_years{year}")

appended (2020-12-11, Sehr geehr) to grouped_years2020
appended (2012-10-26, Frau Präsi) to grouped_years2012
appended (2006-02-10, Frau Präsi) to grouped_years2006
appended (1994-01-21, Frau Präsi) to grouped_years1994
appended (2022-03-23, Frau Präsi) to grouped_years2022
appended (2001-12-13, Selbstvers) to grouped_years2001
appended (1993-01-22, Ich schwör) to grouped_years1993
appended (2015-07-03, Ich kann S) to grouped_years2015
appended (1997-06-12, Antrag der) to grouped_years1997
appended (1996-11-26, Meine sehr) to grouped_years1996
appended (1991-05-14, Fragestund) to grouped_years1991
appended (2001-09-12, Herr Präsi) to grouped_years2001
appended (2018-06-27, Dr. Gottfr) to grouped_years2018
appended (2018-11-20, Herr Präsi) to grouped_years2018
appended (2005-01-27, Frau Präsi) to grouped_years2005
appended (1998-06-17, Frau Präsi) to grouped_years1998
appended (1996-01-31, Frau Kolle) to grouped_years1996
appended (2014-03-20, Herr Präsi) to grouped_years2014
appended (

KeyError: '1990'

Zum Debuggen habe ich mir die Dimension des NumPy-Arrays für das Jahr 1991 ausgeben lassen. Der Output ist folgend zu interpretieren: Es gibt 2 Spalten, da jede Zeile aus dem Datum und dem Text (als String) besteht und davon insgesamt 51 Datums-Text-Paare (51 Zeilen).

In [5]:
print(np.array(grouped_years['1991']).shape)

(51, 2)


Hier werden alle Protokolle aus den jeweils gleichen Jahren zu einer einzigen Zeichenkette zusammengefügt.

In [6]:
 all_speeches = ''
for date, protocol in grouped_years[str(years[0])]:
        all_speeches+=protocol

Diese Zelle ist für das Diagramm am Ende fast am wichtigsten, denn hier wird die Häufigkeit jedes Wortes in den Protokollen für jedes Jahr berechnet. Das Resultat in der Variable counts_for_years ist für jedes Jahr eine Liste von Paaren, die aus einem Wort und seiner Häufigkeit bestehen.

In [7]:
counts_for_years =  {str(year):[] for year in years}
for year in tqdm(years):
    vectorizer = CountVectorizer()
    all_speeches = ''
    for date, protocol in grouped_years[str(year)]:
        all_speeches+=protocol
    count_matrix = vectorizer.fit_transform([all_speeches])
    count_array = count_matrix.toarray()
    word_counts = count_array[0]
    feature_names = vectorizer.get_feature_names_out()
    word_count_pairs = list(zip(feature_names, word_counts))
    counts_for_years[str(year)] = word_count_pairs

100%|██████████| 34/34 [01:40<00:00,  2.95s/it]


Hier gebe ich diese Wörter aus. Dabei habe ich mich beispielhaft auf die 20 am meisten vorkommenden Wörter des Jahres 2022 beschränkt. (Natürlich sind es fast ausschließlich Stopwords).

In [8]:
vectorizer = CountVectorizer()
all_speeches_2019 = ''
for date, protocol in grouped_years['2022']:
    all_speeches_2019+=protocol

count_matrix = vectorizer.fit_transform([all_speeches_2019])

feature_names = vectorizer.get_feature_names_out()

count_array = count_matrix.toarray()

word_counts = count_array[0]

word_count_pairs = list(zip(feature_names, word_counts))

sorted_word_count_pairs = sorted(word_count_pairs, key=lambda x: x[1], reverse=True)
sorted_word_count_pairs[:20]

[('der', 109311),
 ('die', 101399),
 ('und', 95673),
 ('das', 54804),
 ('wir', 51915),
 ('in', 48311),
 ('ist', 41457),
 ('sie', 40145),
 ('bei', 38038),
 ('es', 32110),
 ('nicht', 31441),
 ('auch', 30776),
 ('zu', 29581),
 ('den', 29560),
 ('dass', 26595),
 ('für', 25512),
 ('ich', 24047),
 ('von', 23099),
 ('haben', 21994),
 ('beifall', 21889)]

Diese Zelle erstellt ein Diagramm, was interaktiv, je nach Userinput, eine andere Linie anzeigt.

Die Funktion get_timecounts case ist hier insensitive. Zuvor war sie das nicht und es wurden verschiedene Anzahlen für Wörter angegeben, die ich klein geschrieben habe, als wenn ich sie groß geschrieben hätte. Ich muss die eingegebenen Wörter also normalisieren (Lowercasing) bevor ich sie zählen und anschließend im Diagramm anzeigen lassen kann. 

Betrachtet man den Graph beim Inputtoken "Albrecht", sieht man einen Lineplot mit mehreren Maximums und Minimums. 1992 wurde der Name insgesamt über 40 mal gesagt und 2004 fast 60 mal. Das globale Maxmimum war 2021 mit 63 Erwähnungen. Dieses wird auch grün markiert und mit der Anzahl beschrieben. 2011 war das globale Minimum mit einer Erwähnung. Man kann dies mit allen möglichen Wörtern probieren und bekommt ein interessantes Ergebnis zurück. 
(Nach Eingabe des Tokens muss man einfach kurz warten und der Graph erscheint von allein.)

Die Details für die Erstellung des Graphs habe ich im Code kommentiert.

In [25]:
def get_timecounts_for_token_case_insensitive(token):
    # konvertiere token zu lowercase
    token = token.lower()
    word_timevector = []
    for key, word_count_pairs in counts_for_years.items():
        word_count_dict = {word.lower(): count for word, count in word_count_pairs}
        word_timevector.append(word_count_dict.get(token, 0))
    return word_timevector

def plot_time_series(token):
    if not token:
        print("Please enter a valid token.")
        return
    
    #Normalisierung der Textdaten: Lowercasing 
    outarr = get_timecounts_for_token_case_insensitive(token)
    if not any(outarr): 
        print(f"The token '{token}' is not found in the dataset.")
        return
    
    data = pd.DataFrame({'Year': list(years), 'Value': outarr})

    # hier wurden meine aesthetics für das Diagramm definiert
    sns.set(style='whitegrid', palette='deep', font_scale=1.5, rc={"lines.linewidth": 2.5})

    # Erstellung des LinePlots
    plt.figure(figsize=(14, 8))
    ax = sns.lineplot(x='Year', y='Value', data=data, color='royalblue', linewidth=2.5)

    # Finde die min and max values
    min_value = data['Value'].min()
    max_value = data['Value'].max()
    min_year = data[data['Value'] == min_value]['Year'].values[0]
    max_year = data[data['Value'] == max_value]['Year'].values[0]

    # min und max values farbig markieren
    ax.plot(min_year, min_value, 'o', markersize=10, color='red')
    ax.plot(max_year, max_value, 'o', markersize=10, color='green')

    #
    ax.set_title(f'Time Graph for "{token}" from 1991 to 2024', fontsize=20, weight='bold', pad=20, loc='left', 
                 family='sans-serif', color='darkblue', style='italic')
    ax.set_xlabel('Year', fontsize=16)
    ax.set_ylabel('Value', fontsize=16)
    ax.grid(True, which='both', linestyle='--', linewidth=0.5)

    # Annotate the min and max points
    ax.annotate(f'Min: {min_value}', xy=(min_year, min_value), xytext=(min_year+1, min_value+5),
                arrowprops=dict(facecolor='black', shrink=0.05), fontsize=14, weight='bold', color='red')
    ax.annotate(f'Max: {max_value}', xy=(max_year, max_value), xytext=(max_year+1, max_value+5),
                arrowprops=dict(facecolor='black', shrink=0.05), fontsize=14, weight='bold', color='green')

    plt.show()

# definiert das Eingabefenster für den Userinput
token_widget = widgets.Text(value='', description='Token:', placeholder='Enter a token')
interact(plot_time_series, token=token_widget)

interactive(children=(Text(value='', description='Token:', placeholder='Enter a token'), Output()), _dom_class…

<function __main__.plot_time_series(token)>